# 第 5 周练习 —— 基于 RAG 的技术文档助手

用 **Gradio** 搭一个 **RAG（Retrieval Augmented Generation，检索增强生成）** 系统，回答虚构云平台 **CloudLLM** 的技术文档问题。

## 功能特性

- 用字典写出合成技术文档，并落到本地 Markdown 文件
- 切块（chunk）并生成向量嵌入（embedding）
- 按用户问题检索相关上下文
- 走完整 RAG 管线生成回答
- 交互式 Gradio 聊天界面

## 使用场景

面向虚构云平台「CloudLLM」的技术文档助手——适合练 Week 5：切块策略、本地嵌入、Chroma、检索与生成。

## 怎么跑

1. 准备 `.env`：设置 `OPENAI_API_KEY`（支持 OpenRouter 的 `sk-or-` 前缀或直连 OpenAI）
2. 从上到下运行单元格：先生成知识库，再建向量库，最后 `app.launch()`


In [ ]:
# ========== 导入：标准库 + OpenAI + Gradio + LangChain 组件 ==========

# 标准库 os：读环境变量（API Key）
import os
# 标准库 json：本练习导入备用（当前单元格后续逻辑未强制用到）
import json
# Path：用面向对象方式拼路径、递归扫 .md
from pathlib import Path
# load_dotenv：从 .env 加载密钥，避免写进代码
from dotenv import load_dotenv
# OpenAI 官方客户端：chat.completions 生成回答
from openai import OpenAI
# Gradio：搭 Blocks / ChatInterface UI
import gradio as gr

# ---------- LangChain：切块、本地嵌入、向量库、Document ----------
# 递归字符切块器
from langchain_text_splitters import RecursiveCharacterTextSplitter
# HuggingFace 嵌入：可本机跑，无需付费 API
from langchain_huggingface import HuggingFaceEmbeddings
# Chroma 向量库封装
from langchain_chroma import Chroma
# Document：page_content + metadata 的标准载体
from langchain_core.documents import Document


In [ ]:
# ========== 环境与客户端：区分 OpenRouter / 直连 OpenAI ==========

# 加载 .env；override=True 覆盖已存在的同名环境变量
load_dotenv(override=True)
# 读取 OpenAI / OpenRouter 共用的密钥环境变量名
openai_api_key = os.getenv('OPENAI_API_KEY')

# 冒烟检查：有密钥只打印前缀
if openai_api_key:
    print(f"API Key exists and begins {openai_api_key[:8]}")
else:
    print("API Key not set")

# Setup client (works with OpenRouter or direct OpenAI)
# 若密钥以 sk-or- 开头，走 OpenRouter 兼容端点
if openai_api_key and openai_api_key.startswith('sk-or-'):
    # 显式指定 base_url 指向 OpenRouter
    client = OpenAI(
        api_key=openai_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    # OpenRouter 侧选用的模型 id
    MODEL = "gpt-4.1-mini"
    print("Using OpenRouter")
else:
    # 默认：官方 OpenAI（从环境变量读密钥）
    client = OpenAI()
    # 直连时的模型 id
    MODEL = "gpt-4o-mini"
    print("Using OpenAI directly")

# ---------- 路径配置：向量库目录与知识库目录 ----------
# Chroma 持久化目录名
DB_NAME = "cloudllm_vector_db"
# 合成 Markdown 知识库根目录
KNOWLEDGE_BASE_DIR = "cloudllm_knowledge_base"


In [ ]:
# ========== 合成知识库：写出虚构 CloudLLM 文档，供后面做 RAG ==========
# 说明：下面多段英文 Markdown 是「可检索知识」正文，必须保持原文（prompt/内容字符串不翻译）

# CloudLLM - A fictional cloud platform for LLM deployment
KNOWLEDGE_BASE = {
    "products": {
        "cloudllm_inference.md": """
# CloudLLM Inference

CloudLLM Inference is our flagship product for deploying and serving large language models at scale.

## Features
- **Auto-scaling**: Automatically scales from 0 to 1000+ instances based on traffic
- **Multi-model support**: Deploy GPT, Claude, Llama, Mistral, and custom models
- **Low latency**: Average response time under 100ms for most models
- **Cost optimization**: Pay only for what you use with per-token billing

## Pricing
- Starter: $0.001 per 1K tokens (input) / $0.002 per 1K tokens (output)
- Pro: $0.0008 per 1K tokens with volume discounts
- Enterprise: Custom pricing with dedicated infrastructure

## Getting Started
1. Create a CloudLLM account at dashboard.cloudllm.io
2. Generate an API key in Settings > API Keys
3. Install the SDK: `pip install cloudllm`
4. Make your first request:

```python
from cloudllm import Client
client = Client(api_key="your-api-key")
response = client.inference.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
```
""",
        "cloudllm_embeddings.md": """
# CloudLLM Embeddings

CloudLLM Embeddings provides high-quality vector embeddings for semantic search and RAG applications.

## Supported Models
- **text-embedding-3-large**: 3072 dimensions, best quality
- **text-embedding-3-small**: 1536 dimensions, cost-effective
- **all-MiniLM-L6-v2**: 384 dimensions, fastest

## Pricing
- text-embedding-3-large: $0.00013 per 1K tokens
- text-embedding-3-small: $0.00002 per 1K tokens
- all-MiniLM-L6-v2: Free (self-hosted option available)

## Usage Example
```python
from cloudllm import Client
client = Client(api_key="your-api-key")

embeddings = client.embeddings.create(
    model="text-embedding-3-small",
    input=["Hello world", "How are you?"]
)
```

## Best Practices
- Batch requests for better throughput (up to 100 texts per request)
- Use text-embedding-3-small for most use cases
- Cache embeddings to reduce costs
""",
        "cloudllm_vectordb.md": """
# CloudLLM VectorDB

CloudLLM VectorDB is a fully managed vector database for storing and querying embeddings.

## Features
- **Managed service**: No infrastructure to manage
- **High performance**: Sub-10ms query latency
- **Scalable**: Supports billions of vectors
- **Metadata filtering**: Filter results by metadata fields

## Pricing
- Starter: Free up to 100K vectors
- Pro: $0.25 per 1M vectors/month
- Enterprise: Custom pricing

## Operations
```python
from cloudllm import Client
client = Client(api_key="your-api-key")

# Create collection
collection = client.vectordb.create_collection("my-docs")

# Upsert vectors
collection.upsert(
    ids=["doc1", "doc2"],
    embeddings=[[0.1, 0.2, ...], [0.3, 0.4, ...]],
    documents=["Hello world", "How are you?"],
    metadatas=[{"source": "web"}, {"source": "pdf"}]
)

# Query
results = collection.query(
    query_embeddings=[[0.1, 0.2, ...]],
    n_results=5
)
```
"""
    },
    "guides": {
        "quickstart.md": """
# CloudLLM Quickstart Guide

Get started with CloudLLM in 5 minutes.

## Prerequisites
- Python 3.8+
- CloudLLM account (sign up at cloudllm.io)
- API key

## Installation
```bash
pip install cloudllm
```

## Authentication
Set your API key as an environment variable:
```bash
export CLOUDLLM_API_KEY="your-api-key"
```

Or pass it directly:
```python
from cloudllm import Client
client = Client(api_key="your-api-key")
```

## Your First Request
```python
response = client.inference.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)
```

## Next Steps
- Explore the [API Reference](/docs/api)
- Try [CloudLLM Embeddings](/docs/embeddings)
- Build a [RAG Application](/docs/rag-tutorial)
""",
        "rag_tutorial.md": """
# Building RAG Applications with CloudLLM

Learn how to build a Retrieval Augmented Generation (RAG) application.

## What is RAG?
RAG combines document retrieval with LLM generation to provide accurate, contextual answers.

## Architecture
1. **Indexing**: Chunk documents, create embeddings, store in vector DB
2. **Retrieval**: Find relevant chunks for a user query
3. **Generation**: Use retrieved context to generate an answer

## 步骤 1： Index Documents
```python
from cloudllm import Client

client = Client()
collection = client.vectordb.create_collection("my-docs")

# Chunk and embed documents
for doc in documents:
    chunks = chunk_text(doc, chunk_size=500)
    embeddings = client.embeddings.create(model="text-embedding-3-small", input=chunks)
    collection.upsert(ids=chunk_ids, embeddings=embeddings, documents=chunks)
```

## 步骤 2： Query and Generate
```python
def answer_question(question):
    # Get query embedding
    query_emb = client.embeddings.create(model="text-embedding-3-small", input=[question])
    
    # Retrieve relevant chunks
    results = collection.query(query_embeddings=query_emb, n_results=5)
    context = "\n\n".join(results.documents[0])
    
    # Generate answer
    response = client.inference.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content
```

## Best Practices
- Use chunk sizes of 500-1000 characters with 100-200 overlap
- Retrieve 3-5 relevant chunks per query
- Include source citations in responses
- Consider query rewriting for complex questions
"""
    },
    "company": {
        "about.md": """
# About CloudLLM

CloudLLM is a leading cloud platform for deploying and scaling large language models.

## Our Mission
To make AI accessible to every developer and organization.

## Founded
2023 in San Francisco, CA

## Team
- **Sarah Chen** - CEO & Co-founder (ex-Google AI)
- **Marcus Johnson** - CTO & Co-founder (ex-OpenAI)
- **Priya Patel** - VP Engineering (ex-AWS)
- 50+ engineers and researchers

## Customers
- 10,000+ developers
- 500+ enterprise customers
- Fortune 500 companies in finance, healthcare, and tech

## Contact
- Email: support@cloudllm.io
- Twitter: @cloudllm
- GitHub: github.com/cloudllm
""",
        "support.md": """
# CloudLLM Support

## Getting Help

### Documentation
Visit docs.cloudllm.io for comprehensive guides and API reference.

### Community
- Discord: discord.gg/cloudllm
- GitHub Discussions: github.com/cloudllm/discussions

### Email Support
- Free tier: community@cloudllm.io (48-hour response)
- Pro tier: support@cloudllm.io (24-hour response)
- Enterprise: Dedicated support engineer, 1-hour response SLA

## Common Issues

### Rate Limiting
Default limits: 1000 requests/minute for Pro, 100 for Starter.
Contact sales for higher limits.

### API Key Issues
- Regenerate keys at dashboard.cloudllm.io/settings/api-keys
- Keys starting with `cllm_` are valid
- Never share keys in public repositories

### Billing
View usage at dashboard.cloudllm.io/billing
Set spending alerts in Settings > Billing > Alerts
"""
    }
}

# 把内存里的 KNOWLEDGE_BASE 落到磁盘目录，形成可被 Path.rglob 扫描的知识库
def create_knowledge_base():
    """Create the knowledge base files."""
    # 知识库根目录（与上面 KNOWLEDGE_BASE_DIR 一致）
    base_path = Path(KNOWLEDGE_BASE_DIR)
    
    # 外层：按类别（products / company / support 等）建子目录
    for category, files in KNOWLEDGE_BASE.items():
        # 类别目录：不存在则连同父目录一起创建
        category_path = base_path / category
        category_path.mkdir(parents=True, exist_ok=True)
        
        # 内层：每个文件名对应一段 Markdown 正文
        for filename, content in files.items():
            # 拼出完整文件路径
            file_path = category_path / filename
            # 写入文件：strip() 去掉三引号字符串两端空白
            with open(file_path, "w") as f:
                f.write(content.strip())
    
    # 打印落盘位置，方便核对
    print(f"Knowledge base created at {KNOWLEDGE_BASE_DIR}/")
    
# 立即执行：生成 knowledge base 文件
create_knowledge_base()


In [ ]:
# ========== 加载文档并切块：rglob → Document → RecursiveCharacterTextSplitter ==========

def load_documents():
    """Load all markdown files from knowledge base."""
    # 收集所有 Document
    documents = []
    # 知识库根路径
    base_path = Path(KNOWLEDGE_BASE_DIR)
    
    # 递归找出所有 .md
    for md_file in base_path.rglob("*.md"):
        # 读入全文
        with open(md_file, "r") as f:
            content = f.read()
        
        # Create document with metadata：来源路径、类别目录名、文件名
        doc = Document(
            page_content=content,
            metadata={
                "source": str(md_file),
                "category": md_file.parent.name,
                "filename": md_file.name
            }
        )
        # 加入列表
        documents.append(doc)
    
    return documents

# Load documents：执行加载
documents = load_documents()
print(f"Loaded {len(documents)} documents")

# Chunk documents：按标题/段落优先的分隔符切块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

# 切出 chunks
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

# Preview a chunk：看一眼内容和 metadata 是否合理
print(f"\nSample chunk:")
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")


In [ ]:
# ========== 建向量库：本地 HuggingFace 嵌入 + Chroma 持久化 ==========

# Use HuggingFace embeddings (free, runs locally)
# all-MiniLM-L6-v2：轻量句向量模型；device=cpu 强制 CPU
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Create Chroma vector store：把 chunks 嵌入后写入 DB_NAME 目录
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

# 确认条数与落盘位置
print(f"Vector store created with {len(chunks)} vectors")
print(f"Persisted to {DB_NAME}/")


In [ ]:
# ========== 创建 Retriever，并用测试问题做一次检索冒烟 ==========

# similarity 检索；k=4 表示每次取最相关的 4 个 chunk
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Return top 4 most relevant chunks
)

# Test retrieval：固定英文查询，检验检索是否命中文档
test_query = "How do I get started with CloudLLM?"
results = retriever.invoke(test_query)

# 打印查询与命中块预览
print(f"Query: {test_query}")
print(f"\nRetrieved {len(results)} chunks:")
for i, doc in enumerate(results, 1):
    # 显示来源文件名 + 正文前 200 字符
    print(f"\n--- Chunk {i} ({doc.metadata['filename']}) ---")
    print(doc.page_content[:200] + "...")


In [ ]:
# ========== RAG 管线：检索拼 context → system/user messages → chat.completions ==========

# System Prompt：约束模型只基于文档上下文回答（英文正文保持原样）
SYSTEM_PROMPT = """
You are a helpful technical support assistant for CloudLLM, a cloud platform for deploying LLMs.

Guidelines:
- Answer questions based on the provided context
- Be concise but thorough
- Include code examples when relevant
- If the context doesn't contain enough information, say so
- Suggest relevant documentation pages when appropriate

Context from CloudLLM documentation:
{context}
"""

def answer_question(question: str, history: list) -> str:
    """Answer a question using RAG pipeline."""
    
    # Retrieve relevant documents：用当前问题去检索
    docs = retriever.invoke(question)
    
    # Build context from retrieved documents：带上 Source 文件名，便于模型引用
    context_parts = []
    for doc in docs:
        source = doc.metadata.get('filename', 'unknown')
        context_parts.append(f"[Source: {source}]\n{doc.page_content}")
    
    # 用分隔线把多块拼成一段 context
    context = "\n\n---\n\n".join(context_parts)
    
    # Create messages：system 含上下文，user 是原问题
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question}
    ]
    
    # Generate response：较低 temperature，限制 max_tokens
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.3,
            max_tokens=1000
        )
        # 取第一条 choice 的文本
        return response.choices[0].message.content
    except Exception as e:
        # 失败时返回错误字符串（前缀 Error: 保持原样）
        return f"Error: {str(e)}"


# Test the RAG pipeline：端到端试问一句
test_question = "What is CloudLLM Inference and how much does it cost?"
answer = answer_question(test_question, [])
print(f"Q: {test_question}")
print(f"\nA: {answer}")


In [ ]:
# ========== Gradio UI：示例问题 + ChatInterface + 说明 Markdown ==========

# 点击即可填入的示例问题（英文问句保持原样，与知识库内容对齐）
EXAMPLE_QUESTIONS = [
    "How do I get started with CloudLLM?",
    "What embedding models are available?",
    "How much does CloudLLM Inference cost?",
    "How do I build a RAG application?",
    "Who founded CloudLLM?",
    "How do I contact support?",
    "What is the rate limit for the API?"
]

# Soft 主题的 Blocks；title 会出现在浏览器标签
with gr.Blocks(title="CloudLLM Documentation Assistant", theme=gr.themes.Soft()) as app:
    # 顶部说明（界面文案字符串保持原样）
    gr.Markdown("""
    # CloudLLM Documentation Assistant
    
    Ask questions about CloudLLM products, pricing, and usage.
    
    This assistant uses RAG (Retrieval Augmented Generation) to find relevant 
    documentation and provide accurate answers.
    """)
    
    # 聊天组件：回调 answer_question；messages 格式；挂上 examples
    chatbot = gr.ChatInterface(
        fn=answer_question,
        type="messages",
        examples=EXAMPLE_QUESTIONS,
        title=None
    )
    
    # 页脚：知识库/嵌入/向量库/LLM 一句话说明（字符串保持原样）
    gr.Markdown("""
    ### About This Demo
    - **Knowledge Base**: 7 markdown documents about CloudLLM (fictional)
    - **Embeddings**: HuggingFace all-MiniLM-L6-v2 (384 dimensions)
    - **Vector Store**: Chroma (local)
    - **LLM**: GPT-4.1-mini via OpenRouter
    """)


In [ ]:
# ========== 启动 Gradio 应用 ==========
# Launch the app：阻塞运行，浏览器打开本地 UI
app.launch()


## 技术说明（教学向）

### RAG 管线一览

1. **文档加载（Document Loading）**：从知识库目录读 Markdown
2. **切块（Chunking）**：约 500 字符一块，重叠 100 字符
3. **嵌入（Embedding）**：用 `all-MiniLM-L6-v2` 把块变成向量
4. **存储（Storage）**：写入本地 **Chroma** 向量库
5. **检索（Retrieval）**：每次查询取 top-4 相似块
6. **生成（Generation）**：LLM 基于检索到的上下文回答

### 第 5 周关键概念

- **切块策略**：`RecursiveCharacterTextSplitter` 尽量尊重标题/段落结构
- **重叠（Overlap）**：避免答案刚好落在块边界时丢上下文
- **嵌入模型**：HuggingFace 模型免费、可本机跑
- **向量库**：Chroma 把向量持久化到磁盘
- **Retriever**：LangChain 对相似度搜索的抽象
- **System Prompt**：教模型如何使用 context

### 可以怎么扩展

- 往知识库加更多文档
- 做查询改写（query rewriting）提升召回
- 按类别/日期等做 metadata 过滤
- 加重排序（reranking）提高相关性
- 加评估指标（MRR、NDCG 等）
